In [48]:
import numpy as np
import pandas as pd

from pathlib import Path
import re
import os 
import sys
import numpy.core
from IPython.display import display

sys.modules["numpy._core"] = numpy.core
sys.modules["numpy._core.multiarray"] = numpy.core.multiarray
sys.modules["numpy._core.numeric"] = numpy.core.numeric

In [49]:
PICKLE_PATH = Path(
    "/gpfs01/berens/data/data/Eyewire2/database-exports/clf_rgcs_and_acs/"
    "df_rgcs_and_acs_2026-11-02-19h.pickle"
)

df_features = pd.read_pickle(PICKLE_PATH)

In [50]:
df_features['celltype_conf'].value_counts()

High      2223
Low       1573
Medium     109
Name: celltype_conf, dtype: int64

In [51]:
summary = (
    df_features[df_features["cellclass"] == "RGC"]
    .groupby("celltype")
    .agg(
        total_cells=("celltype", "size"),
        high_conf=("celltype_conf", lambda x: (x == "High").sum()),
        mid_conf=("celltype_conf", lambda x: (x == "Medium").sum()),
        low_conf=("celltype_conf", lambda x: (x == "Low").sum())
    )
    .sort_values("total_cells")
)


display(summary)

,total_cells,high_conf,mid_conf,low_conf
celltype,,,,
m8:X,1,0,0,0
ON vertical OS large RF,1,0,0,0
Separate dendrites and axons,1,0,0,0
m7:9,2,0,0,0
M3,2,1,0,1
EW915,2,0,0,1
ON OS small RF - orientation unknown,2,0,0,0
Other,2,0,0,1
EW82,3,0,0,0


In [52]:

import os
import numpy as np
import pandas as pd


def split_cells(
    df_features,
    heldout_test_types,
    heldout_val_types,
    seed=42,
    split_ratio=0.15,
    min_cells_per_type=5
):
    rng = np.random.default_rng(seed)

    df = df_features[
        df_features["cellclass"] == "RGC"
    ].copy().reset_index(drop=True)

    df["cell_id"] = df["skel_file"].apply(
        lambda x: os.path.splitext(os.path.basename(x))[0]
    )

    heldout_test_types = list(heldout_test_types)
    heldout_val_types = list(heldout_val_types)

    heldout_types = heldout_test_types + heldout_val_types

    # These cells will ONLY go into training
    unknown_type_cells = df[
        df["celltype"].isna()
    ].copy()

    # Only known cell types can be assigned to held-out groups
    known_type_cells = df[
        df["celltype"].notna()
    ].copy()

    # Every cell of these types goes to test
    test_heldout = known_type_cells[
        known_type_cells["celltype"].isin(heldout_test_types)
    ].copy()

    # Every cell of these types goes to validation
    val_heldout = known_type_cells[
        known_type_cells["celltype"].isin(heldout_val_types)
    ].copy()

    # Only non-held-out cells with known celltypes can be sampled
    remaining = known_type_cells[
        ~known_type_cells["celltype"].isin(heldout_types)
    ].copy()

    n_total = len(df)

    target_val = round(n_total * split_ratio)
    target_test = round(n_total * split_ratio)

    n_val_needed = target_val - len(val_heldout)
    n_test_needed = target_test - len(test_heldout)

    if n_val_needed < 0:
        raise ValueError(
            f"All held-out validation cells ({len(val_heldout)}) "
            f"exceed the validation target ({target_val})."
        )

    if n_test_needed < 0:
        raise ValueError(
            f"All held-out test cells ({len(test_heldout)}) "
            f"exceed the test target ({target_test})."
        )

    type_counts = remaining["celltype"].value_counts()

    eligible_types = type_counts[
        type_counts >= min_cells_per_type
    ].index.tolist()

    eligible = remaining[
        remaining["celltype"].isin(eligible_types)
    ].copy()

    # Ensure each eligible type can appear in both splits
    if n_val_needed < len(eligible_types):
        raise ValueError(
            "Not enough validation slots to include every "
            "eligible cell type."
        )

    if n_test_needed < len(eligible_types):
        raise ValueError(
            "Not enough test slots to include every "
            "eligible cell type."
        )

    # --------------------------------------------------
    # 7. Reserve one cell per eligible type for each split
    # --------------------------------------------------

    val_indices = []
    test_indices = []

    for cell_type in eligible_types:

        type_data = eligible[
            eligible["celltype"] == cell_type
        ].copy()

        # Shuffle within confidence groups
        high = type_data[
            type_data["celltype_conf"] == "High"
        ]

        other = type_data[
            type_data["celltype_conf"] != "High"
        ]

        high_indices = high.index.to_numpy().copy()
        other_indices = other.index.to_numpy().copy()

        rng.shuffle(high_indices)
        rng.shuffle(other_indices)

        ordered_indices = np.concatenate([
            high_indices,
            other_indices
        ])

        if len(ordered_indices) < 2:
            raise ValueError(
                f"{cell_type} does not have enough cells "
                "for both validation and test."
            )

        # Reserve two distinct cells
        val_indices.append(ordered_indices[0])
        test_indices.append(ordered_indices[1])

    # --------------------------------------------------
    # 8. Fill remaining validation and test slots
    # --------------------------------------------------

    used_indices = set(val_indices) | set(test_indices)

    available = eligible.loc[
        ~eligible.index.isin(used_indices)
    ].copy()

    # Prioritize High, then Medium, Low, and missing confidence
    confidence_order = ["High", "Medium", "Low"]

    candidate_groups = []

    for confidence in confidence_order:
        group = available[
            available["celltype_conf"] == confidence
        ].index.to_numpy().copy()

        rng.shuffle(group)
        candidate_groups.append(group)

    # Include missing or other confidence values last
    other_confidence = available[
        ~available["celltype_conf"].isin(confidence_order)
    ].index.to_numpy().copy()

    rng.shuffle(other_confidence)
    candidate_groups.append(other_confidence)

    candidates = np.concatenate(candidate_groups)

    n_val_extra = n_val_needed - len(val_indices)
    n_test_extra = n_test_needed - len(test_indices)

    n_extra_total = n_val_extra + n_test_extra

    if n_extra_total > len(candidates):
        raise ValueError(
            "Not enough remaining cells to reach the requested "
            "validation and test sizes."
        )

    extra_indices = candidates[:n_extra_total]

    val_extra = extra_indices[:n_val_extra]
    test_extra = extra_indices[n_val_extra:]

    val_indices.extend(val_extra.tolist())
    test_indices.extend(test_extra.tolist())

    # --------------------------------------------------
    # 9. Construct validation and test
    # --------------------------------------------------

    val_sampled = remaining.loc[val_indices].copy()
    test_sampled = remaining.loc[test_indices].copy()

    val = pd.concat(
        [val_heldout, val_sampled],
        axis=0
    )

    test = pd.concat(
        [test_heldout, test_sampled],
        axis=0
    )

    # --------------------------------------------------
    # 10. Training = all unassigned cells
    # --------------------------------------------------

    assigned_indices = set(val.index) | set(test.index)

    train = df.loc[
        ~df.index.isin(assigned_indices)
    ].copy()

    # This includes ALL cells with missing celltype
    assert unknown_type_cells.index.isin(train.index).all()

    # --------------------------------------------------
    # 11. Shuffle splits
    # --------------------------------------------------

    train = train.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    val = val.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    test = test.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    # --------------------------------------------------
    # 12. Verify no cells are missing or duplicated
    # --------------------------------------------------

    train_ids = set(train["cell_id"])
    val_ids = set(val["cell_id"])
    test_ids = set(test["cell_id"])

    original_ids = set(df["cell_id"])

    assert train_ids.isdisjoint(val_ids), "Train/val overlap!"
    assert train_ids.isdisjoint(test_ids), "Train/test overlap!"
    assert val_ids.isdisjoint(test_ids), "Val/test overlap!"

    all_assigned_ids = train_ids | val_ids | test_ids

    missing_ids = original_ids - all_assigned_ids
    extra_ids = all_assigned_ids - original_ids

    assert not missing_ids, f"Missing cells: {len(missing_ids)}"
    assert not extra_ids, f"Unexpected cells: {len(extra_ids)}"

    assert len(train) + len(val) + len(test) == len(df)

    # All missing-celltype cells must be in training
    unknown_ids = set(unknown_type_cells["cell_id"])

    assert unknown_ids.issubset(train_ids)
    assert not unknown_ids.intersection(val_ids)
    assert not unknown_ids.intersection(test_ids)

    # Held-out types must only appear in their designated split
    assert not train["celltype"].isin(heldout_types).any()

    assert not val["celltype"].isin(heldout_test_types).any()
    assert not test["celltype"].isin(heldout_val_types).any()

    # Check that every held-out cell is present
    assert set(test_heldout["cell_id"]).issubset(test_ids)
    assert set(val_heldout["cell_id"]).issubset(val_ids)

    # --------------------------------------------------
    # 13. Print summary
    # --------------------------------------------------

    print("\nSPLIT SUMMARY")
    print("----------------------------")
    print(f"Total RGC cells: {len(df)}")
    print(f"Train:      {len(train)} ({len(train)/len(df):.2%})")
    print(f"Validation: {len(val)} ({len(val)/len(df):.2%})")
    print(f"Test:       {len(test)} ({len(test)/len(df):.2%})")

    print("\nCells with missing celltype:")
    print(f"Assigned to training: {len(unknown_type_cells)}")

    print("\nHeld-out validation types:", heldout_val_types)
    print("Held-out test types:", heldout_test_types)

    print("\nHeld-out validation counts:")
    print(
        val[val["celltype"].isin(heldout_val_types)]
        .groupby("celltype").size()
    )

    print("\nHeld-out test counts:")
    print(
        test[test["celltype"].isin(heldout_test_types)]
        .groupby("celltype").size()
    )

    return train, val, test, heldout_val_types, heldout_test_types


In [53]:
df_features[df_features["cellclass"] == "RGC"].shape

(3831, 417)

In [54]:
train, val, test, heldout_val_types, heldout_test_types = split_cells(
    df_features,
    heldout_test_types=["ON DS transient", "EW3o-like"],
    heldout_val_types=["EW82wi"],
    seed=42,
    split_ratio=0.15
)


SPLIT SUMMARY
----------------------------
Total RGC cells: 3831
Train:      2681 (69.98%)
Validation: 575 (15.01%)
Test:       575 (15.01%)

Cells with missing celltype:
Assigned to training: 63

Held-out validation types: ['EW82wi']
Held-out test types: ['ON DS transient', 'EW3o-like']

Held-out validation counts:
celltype
EW82wi    33
dtype: int64

Held-out test counts:
celltype
EW3o-like          20
ON DS transient    32
dtype: int64


In [46]:
# def split_cells(
#     df_features,
#     exclude_type_val=None,
#     exclude_type_test=None,
#     seed=42,
#     split_ratio=0.15
# ):
#     rng = np.random.default_rng(seed)

#     df = df_features[
#         (df_features["cellclass"] == "RGC") &
#         (df_features["celltype"].notna())
#     ].copy().reset_index(drop=True)

#     df["cell_id"] = df["skel_file"].apply(
#         lambda x: os.path.splitext(os.path.basename(x))[0]
#     )

#     cell_types = df["celltype"].unique().tolist()

#     # Select 3 cell types for validation
#     if exclude_type_val is None:
#         exclude_type_val = rng.choice(
#             cell_types, size=3, replace=False
#         ).tolist()

#     # Select 3 different cell types for test
#     remaining_types = [
#         t for t in cell_types if t not in exclude_type_val
#     ]

#     if exclude_type_test is None:
#         exclude_type_test = rng.choice(
#             remaining_types, size=3, replace=False
#         ).tolist()

#     heldout_types = exclude_type_val + exclude_type_test

#     # High-confidence cells for validation and test
#     high = df[df["celltype_conf"] == "High"].copy()

#     val_parts = []
#     test_parts = []

#     # 1. Validation: held-out validation types
#     val_heldout = high[
#         high["celltype"].isin(exclude_type_val)
#     ]

#     if len(val_heldout) == 0:
#         raise ValueError("No High-confidence validation cells.")

#     val_parts.append(val_heldout)

#     # 2. Test: held-out test types
#     test_heldout = high[
#         high["celltype"].isin(exclude_type_test)
#     ]

#     if len(test_heldout) == 0:
#         raise ValueError("No High-confidence test cells.")

#     test_parts.append(test_heldout)

#     # 3. Remaining cell types: stratified 15% per cell type
#     for cell_type in cell_types:

#         if cell_type in heldout_types:
#             continue

#         type_data = high[high["celltype"] == cell_type]

#         n = len(type_data)

#         n_val = round(n * split_ratio)
#         n_test = round(n * split_ratio)

#         if n < n_val + n_test:
#             raise ValueError(
#                 f"Not enough High-confidence cells for {cell_type}"
#             )

#         # Sample validation
#         val_sample = type_data.sample(
#             n=n_val,
#             random_state=seed
#         )

#         remaining = type_data.drop(index=val_sample.index)

#         # Sample test from remaining cells
#         test_sample = remaining.sample(
#             n=n_test,
#             random_state=seed
#         )

#         val_parts.append(val_sample)
#         test_parts.append(test_sample)

#     # Combine validation and test
#     val = pd.concat(val_parts)
#     test = pd.concat(test_parts)

#    # Training: exclude held-out types and sampled validation/test cells
#     excluded_indices = val.index.union(test.index)

#     train = df[
#         (~df["celltype"].isin(heldout_types)) &
#         (~df.index.isin(excluded_indices))
#     ].copy()

#     # Shuffle
#     train = train.sample(frac=1, random_state=seed)
#     val = val.sample(frac=1, random_state=seed)
#     test = test.sample(frac=1, random_state=seed)

#     return train, val, test, exclude_type_val, exclude_type_test


In [47]:
# train, val, test, excluded_val, excluded_test = split_cells(df_features)

In [55]:
np.save("train_ids.npy", train["cell_id"].to_numpy(dtype=str))
np.save("val_ids.npy", val["cell_id"].to_numpy(dtype=str))
np.save("test_ids.npy", test["cell_id"].to_numpy(dtype=str))

print("Saved:")
print("Train:", len(train), "cells")
print("Validation:", len(val), "cells")
print("Test:", len(test), "cells")

Saved:
Train: 2681 cells
Validation: 575 cells
Test: 575 cells


Remove cells with less than 1000 nodes after axon removal (during extraction)

In [56]:
cell_id_to_remove = "720575940560435035"

train = train[train["cell_id"] != cell_id_to_remove].copy()
val = val[val["cell_id"] != cell_id_to_remove].copy()
test = test[test["cell_id"] != cell_id_to_remove].copy()

print("Train:", len(train))
print("Val:", len(val))
print("Test:", len(test))

Train: 2680
Val: 575
Test: 575


In [57]:
np.save("train_ids.npy", train["cell_id"].to_numpy(dtype=str))
np.save("val_ids.npy", val["cell_id"].to_numpy(dtype=str))
np.save("test_ids.npy", test["cell_id"].to_numpy(dtype=str))

In [58]:
all_ids = np.concatenate([
    train["cell_id"].to_numpy(dtype=str),
    val["cell_id"].to_numpy(dtype=str),
    test["cell_id"].to_numpy(dtype=str)
])


np.save("all_ids.npy", all_ids)